# 🔀 Agent Orchestration z LangGraph

Budujemy graf decyzyjny do routowania zgloszen IT. Agent klasyfikuje zgloszenie, sprawdza pewnosc, w razie potrzeby siega do bazy wiedzy (RAG), a nastepnie routuje do odpowiedniego zespolu.

```
START → [Klasyfikuj] → {pewnosc > 80%?}
                         ├── TAK → [Routuj] → END
                         └── NIE → [Szukaj w KB] → [Reklasyfikuj] → [Routuj] → END
```

**Czas:** ~45 minut | **Wymagania:** Google Colab (GPU nie wymagane!), klucz API OpenRouter

## 🛠️ 0. Instalacja i konfiguracja

### 📖 Wytlumaczenie:
LangGraph to framework do budowania agentow jako grafow. Wezly (nodes) to kroki, krawedzie (edges) to decyzje. OpenRouter dziala jako backend LLM (ten sam co w poprzednim notebooku).

### 💡 Cwiczenie:
Po instalacji sprawdz wersje: `import langgraph; print(langgraph.__version__)`

In [ ]:
!pip install -q langgraph

import os
import requests
from getpass import getpass

# === Konfiguracja API ===
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL = "meta-llama/llama-3.1-8b-instruct"

# Zaladuj klucz API: os.environ -> Colab Secrets -> getpass
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPENROUTER_API_KEY")
        print("Klucz API zaladowany z Colab Secrets")
    except Exception:
        pass

if not API_KEY:
    API_KEY = getpass("Wklej klucz OpenRouter API: ")

# Test polaczenia
USE_API = False
if API_KEY:
    test_resp = requests.post(
        url=OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": "Powiedz OK"}],
            "max_tokens": 5,
        },
        timeout=15,
    )
    if test_resp.ok:
        USE_API = True
        print(f"Model: {MODEL}")
        print(f"API: dziala!")
    else:
        print(f"API error {test_resp.status_code}: {test_resp.text[:200]}")
        print("Przelaczam na tryb fallback (mock responses)")
else:
    print("Brak klucza API — tryb fallback (mock responses)")
    print("Notebook bedzie dzialal, ale odpowiedzi beda symulowane.")

print("\n✅ LangGraph + OpenRouter gotowe!")

In [ ]:
import json
import requests
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END


def call_llm(prompt, system=""):
    """Call LLM via OpenRouter."""
    if not USE_API:
        return "[FALLBACK]"

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    r = requests.post(
        url=OPENROUTER_URL,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": MODEL,
            "messages": messages,
        },
        timeout=60,
    )

    if not r.ok:
        return "[FALLBACK]"

    return r.json()["choices"][0]["message"]["content"].strip()

---
## 🔹 Krok 1: Definiujemy State — co agent pamięta (~10 min)

### 📖 Wytlumaczenie:
State to "notatnik" agenta. Kazdy wezel (node) czyta ze State i zapisuje do niego. State przeplywa przez caly graf. W LangGraph definiujemy go jako `TypedDict` — jasna struktura, typowane pola.

### 💡 Cwiczenie:
Dodaj pole `processing_time_ms` do State zeby sledzic czas przetwarzania.

In [ ]:
# === Define the agent state ===

class TicketState(TypedDict):
    text: str               # original ticket text
    category: str           # predicted category
    confidence: float       # classification confidence (0-1)
    context: str            # retrieved KB context (for RAG)
    route: str              # final routing destination
    needs_review: bool      # flag for human review
    path: list              # track which nodes were visited

print("✅ TicketState zdefiniowany z 7 polami")
print("   text, category, confidence, context, route, needs_review, path")

---
## 🔹 Krok 2: Wezly (Nodes) — co agent robi w kazdym kroku (~10 min)

### 📖 Wytlumaczenie:
Kazdy wezel to funkcja Pythona. Przyjmuje State, cos robi (np. wywoluje LLM), i zwraca zaktualizowany fragment State. LangGraph sam laczy wyniki.

### 💡 Cwiczenie:
Dodaj wezel `log_ticket` na koncu grafu, ktory zapisuje wynik do listy (symulacja bazy danych).

In [ ]:
# === Knowledge Base ===

KNOWLEDGE_BASE = [
    {"keywords": ["drukarka sieciowa", "printer", "sieciowa nie odpowiada"],
     "content": "Network printers (IP/WiFi): if not responding, issue is network connection. Classify as: Network."},
    {"keywords": ["wifi", "wi-fi", "sieci wifi", "zalogowac do sieci"],
     "content": "WiFi login issues are Network problems, not Account. Classify as: Network."},
    {"keywords": ["haslo", "hasla", "zmiana hasla", "password", "erp"],
     "content": "Password changes in any app (ERP, CRM): root cause is account management. Classify as: Account."},
    {"keywords": ["dostep", "folderu", "po zmianie hasla", "access", "after password"],
     "content": "Access issues after password change: AD sync needed. Root cause: password. Classify as: Account."},
    {"keywords": ["antywirus", "antywirusowy", "antivirus", "blokuje"],
     "content": "Antivirus blocking software: software conflict. Classify as: Software."},
]

TEAM_ROUTING = {
    "Hardware": "hardware-team@company.com (Piętro 2, pokój 204)",
    "Network": "network-ops@company.com (NOC, pokój 101)",
    "Software": "software-support@company.com (Piętro 3, pokój 312)",
    "Account": "identity-team@company.com (Piętro 1, pokój 108)",
}

print("✅ Baza wiedzy: {} artykulow".format(len(KNOWLEDGE_BASE)))
print("✅ Routing: {} zespolow".format(len(TEAM_ROUTING)))

In [ ]:
# === Node 1: Classify ticket ===

# Fallback classifications for when API is not available
FALLBACK_CLASSIFICATIONS = {
    "Drukarka nie drukuje": ("Hardware", 0.95),
    "Nie moge sie zalogowac do poczty": ("Account", 0.90),
    "VPN nie laczy sie z siecia firmowa": ("Network", 0.92),
    "Aplikacja CRM zawiesza sie po aktualizacji": ("Software", 0.88),
    "Drukarka sieciowa nie odpowiada": ("Hardware", 0.55),     # low confidence!
    "Nie moge zalogowac sie do sieci WiFi": ("Account", 0.50),  # low confidence!
    "System ERP nie pozwala na zmiane hasla": ("Software", 0.45),# low confidence!
    "Monitor migocze i komputer sie wylacza": ("Hardware", 0.93),
    "Brak dostepu do folderu sieciowego po zmianie hasla": ("Network", 0.48),  # low!
    "Program antywirusowy blokuje aktualizacje systemu": ("Software", 0.87),
}

def classify_node(state: TicketState) -> dict:
    """Node: classify the ticket using LLM."""
    text = state["text"]

    if USE_API:
        prompt = (
            "Classify this IT ticket. Return ONLY JSON:\n"
            '{"category": "Hardware|Network|Software|Account", "confidence": 0.0-1.0}\n\n'
            f'Ticket: "{text}"\nJSON:'
        )
        response = call_llm(prompt, system="Return only valid JSON.")
        try:
            start = response.index("{")
            end = response.rindex("}") + 1
            result = json.loads(response[start:end])
            category = result.get("category", "Unknown")
            confidence = float(result.get("confidence", 0.5))
        except:
            category, confidence = "Unknown", 0.3
    else:
        category, confidence = FALLBACK_CLASSIFICATIONS.get(text, ("Unknown", 0.5))

    return {
        "category": category,
        "confidence": confidence,
        "path": state.get("path", []) + ["classify"],
    }

print("✅ Node: classify_node")

In [ ]:
# === Node 2: Search knowledge base ===

def search_kb_node(state: TicketState) -> dict:
    """Node: search knowledge base for relevant context."""
    text = state["text"].lower()
    relevant = []
    for entry in KNOWLEDGE_BASE:
        if any(kw in text for kw in entry["keywords"]):
            relevant.append(entry["content"])

    context = " | ".join(relevant) if relevant else "No relevant KB articles found."
    return {
        "context": context,
        "path": state.get("path", []) + ["search_kb"],
    }

print("✅ Node: search_kb_node")

In [ ]:
# === Node 3: Reclassify with context (RAG) ===

FALLBACK_RAG = {
    "Drukarka sieciowa nie odpowiada": ("Network", 0.92),
    "Nie moge zalogowac sie do sieci WiFi": ("Network", 0.88),
    "System ERP nie pozwala na zmiane hasla": ("Account", 0.90),
    "Brak dostepu do folderu sieciowego po zmianie hasla": ("Account", 0.91),
}

def reclassify_node(state: TicketState) -> dict:
    """Node: reclassify ticket using KB context (RAG)."""
    text = state["text"]
    context = state.get("context", "")

    if USE_API:
        prompt = (
            "Classify this IT ticket using the provided context.\n"
            'Return ONLY JSON: {"category": "Hardware|Network|Software|Account", "confidence": 0.0-1.0}\n\n'
            f'Context: {context}\n\n'
            f'Ticket: "{text}"\nJSON:'
        )
        response = call_llm(prompt, system="Return only valid JSON.")
        try:
            start = response.index("{")
            end = response.rindex("}") + 1
            result = json.loads(response[start:end])
            category = result.get("category", state["category"])
            confidence = float(result.get("confidence", 0.7))
        except:
            category = state["category"]
            confidence = 0.6
    else:
        category, confidence = FALLBACK_RAG.get(text, (state["category"], 0.7))

    return {
        "category": category,
        "confidence": confidence,
        "path": state.get("path", []) + ["reclassify_rag"],
    }

print("✅ Node: reclassify_node")

In [ ]:
# === Node 4: Route to team ===

def route_node(state: TicketState) -> dict:
    """Node: route ticket to the appropriate team."""
    category = state["category"]
    route = TEAM_ROUTING.get(category, "unknown-team@company.com")
    needs_review = state["confidence"] < 0.7

    return {
        "route": route,
        "needs_review": needs_review,
        "path": state.get("path", []) + ["route"],
    }

print("✅ Node: route_node")

---
## 🔹 Krok 3: Krawedzie warunkowe — agent podejmuje decyzje (~5 min)

### 📖 Wytlumaczenie:
Krawedz warunkowa (conditional edge) to punkt decyzyjny w grafie. Na podstawie State agent wybiera, ktora sciezka isc. Tutaj: jesli pewnosc > 80% → routuj od razu. Jesli nie → szukaj w bazie wiedzy i reklasyfikuj.

### 💡 Cwiczenie:
Zmien prog z 0.8 na 0.6 — ile zgloszen teraz idzie sciezka szybka vs RAG?

In [ ]:
# === Conditional edge: confidence check ===

CONFIDENCE_THRESHOLD = 0.8

def check_confidence(state: TicketState) -> Literal["route", "search_kb"]:
    """Decide: if confidence is high → route directly. If low → search KB first."""
    if state["confidence"] >= CONFIDENCE_THRESHOLD:
        return "route"
    else:
        return "search_kb"

print(f"✅ Conditional edge: prog pewnosci = {CONFIDENCE_THRESHOLD}")
print(f"   >= {CONFIDENCE_THRESHOLD} → sciezka szybka (classify → route)")
print(f"   <  {CONFIDENCE_THRESHOLD} → sciezka RAG (classify → search_kb → reclassify → route)")

---
## 🔹 Krok 4: Budujemy graf (~10 min)

### 📖 Wytlumaczenie:
Teraz laczymy wszystko w graf. Definiujemy wezly, krawedzie i kompilujemy. LangGraph tworzy z tego wykonywalny pipeline.

```
START → [classify] → {confidence >= 0.8?}
                       ├── YES → [route] → END
                       └── NO  → [search_kb] → [reclassify] → [route] → END
```

### 💡 Cwiczenie:
Dodaj wezel `human_review` przed `route` dla zgloszen z confidence < 0.5.

In [ ]:
# === Build the graph ===

graph = StateGraph(TicketState)

# Add nodes
graph.add_node("classify", classify_node)
graph.add_node("search_kb", search_kb_node)
graph.add_node("reclassify", reclassify_node)
graph.add_node("route", route_node)

# Add edges
graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", check_confidence, {
    "route": "route",
    "search_kb": "search_kb",
})
graph.add_edge("search_kb", "reclassify")
graph.add_edge("reclassify", "route")
graph.add_edge("route", END)

# Compile!
app = graph.compile()

print("✅ Graf skompilowany!")
print("   Wezly: classify → search_kb → reclassify → route")
print("   Decyzja: confidence >= 0.8 → szybka sciezka")

---
## 🔹 Krok 5: Uruchamiamy graf na 10 zgloszeniach (~5 min)

### 📖 Wytlumaczenie:
Te same 10 zgloszen co we wszystkich poprzednich cwiczeniach. Teraz widzimy, ktora sciezka przeszlo kazde zgloszenie: szybka (2 wezly) czy RAG (4 wezly).

### 💡 Cwiczenie:
Dodaj wlasne zgloszenie i sprawdz, ktora sciezka przejdzie.

In [ ]:
# === Process all 10 test tickets ===

test_tickets = [
    "Drukarka nie drukuje",
    "Nie moge sie zalogowac do poczty",
    "VPN nie laczy sie z siecia firmowa",
    "Aplikacja CRM zawiesza sie po aktualizacji",
    "Drukarka sieciowa nie odpowiada",
    "Nie moge zalogowac sie do sieci WiFi",
    "System ERP nie pozwala na zmiane hasla",
    "Monitor migocze i komputer sie wylacza",
    "Brak dostepu do folderu sieciowego po zmianie hasla",
    "Program antywirusowy blokuje aktualizacje systemu",
]

correct_labels = [
    "Hardware", "Account", "Network", "Software",
    "Network", "Network", "Account",
    "Hardware", "Account", "Software",
]

results = []

for i, (ticket, correct) in enumerate(zip(test_tickets, correct_labels), 1):
    # Run the graph
    initial_state = {
        "text": ticket,
        "category": "",
        "confidence": 0.0,
        "context": "",
        "route": "",
        "needs_review": False,
        "path": [],
    }

    final_state = app.invoke(initial_state)
    results.append(final_state)

    # Display result
    is_correct = final_state["category"] == correct
    icon = "✅" if is_correct else "❌"
    path_str = " → ".join(final_state["path"])
    used_rag = "search_kb" in final_state["path"]
    path_icon = "📚 RAG" if used_rag else "⚡ Fast"

    print(f"{icon} {i:2d}. \"{ticket}\"")
    print(f"    Kategoria: {final_state['category']} (pewnosc: {final_state['confidence']:.0%}) | Poprawna: {correct}")
    print(f"    Sciezka: {path_icon} [{path_str}]")
    print(f"    Routing: {final_state['route']}")
    if final_state["needs_review"]:
        print(f"    ⚠️ Wymaga przegladu ludzkiego")
    print()

---
## 🔹 Krok 6: Wizualizacja wynikow (~5 min)

### 📖 Wytlumaczenie:
Zobaczmy statystyki: ile zgloszen poszlo sciezka szybka, ile przez RAG, i jaka jest dokladnosc kazdej sciezki.

In [ ]:
import matplotlib.pyplot as plt

# Analyze results
fast_path = [r for r in results if "search_kb" not in r["path"]]
rag_path = [r for r in results if "search_kb" in r["path"]]

fast_correct = sum(1 for r, c in zip(results, correct_labels)
                   if "search_kb" not in r["path"] and r["category"] == c)
rag_correct = sum(1 for r, c in zip(results, correct_labels)
                  if "search_kb" in r["path"] and r["category"] == c)

total_correct = sum(1 for r, c in zip(results, correct_labels) if r["category"] == c)

print("📊 Statystyki grafu:\n")
print(f"   ⚡ Sciezka szybka: {len(fast_path)} zgloszen ({fast_correct}/{len(fast_path)} poprawnych)")
print(f"   📚 Sciezka RAG:    {len(rag_path)} zgloszen ({rag_correct}/{len(rag_path)} poprawnych)")
print(f"   🎯 Lacznie:        {total_correct}/{len(results)} = {total_correct/len(results):.0%}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Path distribution
axes[0].bar(["Fast Path", "RAG Path"], [len(fast_path), len(rag_path)],
            color=["#3498db", "#2ecc71"])
axes[0].set_title("Rozklad sciezek", fontsize=13)
axes[0].set_ylabel("Liczba zgloszen")

# 2. Confidence distribution
confidences = [r["confidence"] for r in results]
colors = ["#2ecc71" if r["category"] == c else "#e74c3c"
          for r, c in zip(results, correct_labels)]
axes[1].barh(range(len(test_tickets)), confidences, color=colors)
axes[1].set_yticks(range(len(test_tickets)))
axes[1].set_yticklabels([t[:30] + "..." if len(t) > 30 else t for t in test_tickets], fontsize=8)
axes[1].set_xlabel("Pewnosc")
axes[1].set_title("Pewnosc klasyfikacji (zielony=OK, czerwony=blad)", fontsize=11)
axes[1].axvline(x=CONFIDENCE_THRESHOLD, color="orange", linestyle="--", label=f"Prog={CONFIDENCE_THRESHOLD}")
axes[1].legend()

# 3. Accuracy comparison: fast vs RAG
fast_acc = fast_correct / len(fast_path) if fast_path else 0
rag_acc = rag_correct / len(rag_path) if rag_path else 0
total_acc = total_correct / len(results)
axes[2].bar(["Fast", "RAG", "Total"], [fast_acc, rag_acc, total_acc],
            color=["#3498db", "#2ecc71", "#9b59b6"])
axes[2].set_title("Dokladnosc per sciezka", fontsize=13)
axes[2].set_ylabel("Dokladnosc")
axes[2].set_ylim(0, 1.15)
for j, v in enumerate([fast_acc, rag_acc, total_acc]):
    axes[2].text(j, v + 0.03, f"{v:.0%}", ha="center", fontweight="bold")

plt.suptitle("LangGraph IT Ticket Router — Analiza wynikow", fontsize=14)
plt.tight_layout()
plt.show()

---
## 🎓 Podsumowanie: Wzorce orkiestracji agentow (2026)

### 📖 Wytlumaczenie:

To co zbudowalismy to wzorzec **Router** — najprostszy typ orkiestracji. W produkcji uzywa sie tez:

| Wzorzec | Opis | Przyklad |
|---------|------|----------|
| **Router** | Klasyfikuj → Routuj (nasz graf!) | Helpdesk, triage |
| **Plan-and-Execute** | Jeden model planuje, tani model wykonuje | Redukcja kosztow o 90% |
| **Reflect** | Agent sprawdza swoj wynik i poprawia | QA, recenzje |
| **Human-in-the-Loop** | Agent pyta czlowieka w kluczowych momentach | Finanse, medycyna |

```
Nasz graf:
                    ┌── [Fast: 2 wezly] ──┐
START → [Classify] →│                      │→ [Route] → END
                    └── [RAG: 4 wezly]  ──┘
```

### 💡 Cwiczenie koncowe:
Rozbuduj graf o dodatkowy wezel `human_review` dla zgloszen z pewnoscia < 50%. Wezel powinien wyswietlac pytanie do czlowieka i czekac na input.

> 🏆 *Agenci, ktorzy dzialaja w produkcji, to te nudne: waskie, wyspecjalizowane, gleboko zintegrowane.*